In [1]:
import numpy as np
import pandas as pd
import os
import tifffile
import cv2
from os.path import join, isfile, exists
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
import torch.nn as nn
from torchcam.utils import overlay_mask
from torchcam.methods import SmoothGradCAMpp
import os
from PIL import Image
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import matplotlib.pyplot as plt
import time
import copy
import warnings
import contextlib

In [2]:
def strTmpt(t):
    if t == 1:
        return 't001'
    elif t < 100:
        return 't0'+str(t)
    else:
        return 't'+str(t)

def findNet(t,randCat = 'Actual'):
    tOI = strTmpt(t)
    if randCat == 'Actual':
        nets = os.listdir('TRAIN_ResNets_Trained/Actual')
    else:
        nets = os.listdir('TRAIN_ResNets_Trained/RandCat1')
    for net in nets:
        if tOI in net and not 'RdLR' in net:
            return net 
    
netType = 'ResNet'
#netType = 'TDANet'

In [3]:
#June 25, My Comments: This is a "given specific colony, find what it's time is"
#and not a "load all colony conditions at once", like I had

def get_condition(title,randCat):
    if randCat == 'Actual':
        df = pd.read_csv('tile_conditions.csv')
        return df['conds'][list(df['samps']).index(title)]
        ## Filter the DataFrame based on the given parameters
        ##filtered_df = df[(df['Experiment'].dt.day == int(title.split('_')[0])) & (df['Tile'] == int(title.split('_')[1]))]
        #f#    return condition
        #else:
        #    print('Something weird getting condition for '+title+' when '+ rand)
        #    return None
    else:
        df = pd.read_csv('RAND_RandCat1.csv')
        if title[3] == '0':
            newtitle = title[:3]+title[4]
        else:
            newtitle = title
        return df['Category'][list(df['Colony']).index(newtitle)]
        
        ## Filter the DataFra#e based on the given parameters
        #filtered_df = df[(df['Experiment'].dt.day == int(title.split('_')[0])) & (df['Tile'] == int(title.split('_')[1]))]
        #filtered_df = df[int(df['Colony'].split('_')[0]) == int(title.split('_')[0])\
        #    & int(df['Colony'].split('_')[1]) == int(title.split('_')[1])]
        ## Check if there are any matching rows
        #if len(filtered_df) == 1:
        #    # Retrieve the condition from the first matching row
        #    condition = filtered_df.iloc[0]['Category']
        #    return condition
        #else:
        #    print('Something weird getting condition for '+title+' when '+ rand)
        #    return None

#June 25th, My Comments:
#The Dataset parameter is the directory where we keep all the folders of the images:
#see, for example, on GitHub where she instantiates an object of this class in the code

# Define the dataset class
#August 23: transforming for ripser data,
#         Ripser: indexed by timepoint, then sample
#         Images: indexed by sample, then timepoint
#       so have to redo file extraction

# Define the dataset class
class MyDataset(Dataset):
    def __init__(self, root_dir, randCat, transform=None, timestamp=1):
        self.root_dir = root_dir
        self.transform = transform
        self.images = []
        self.labels = []
        label_map 
        for folder in os.listdir(root_dir):
            #June 25th, My Comments: since my image directory has stuff besides  the '22_..' and '26_..'
            #cell colony image folders, I include this if statement to filter them not
            if len(folder) == 5 and (folder[:2] == '22' or folder[:2] == '26'):
                folder_path = os.path.join(root_dir, folder)
                files = [f for f in os.listdir(folder_path) if isfile(join(folder_path, f))]
                label = get_condition(folder,randCat)
                # print(folder_path, label)
                for file in files:
                    file_path = os.path.join(folder_path, file)
                    match = re.search(r'_t(\d+)_c002', file)
                    if match:
                        number = int(match.group(1))
                        if number == timestamp:
                            # print(file)
                            if file_path.endswith(".png"):
                                img = Image.open(file_path).convert("RGB")
                                name = file.split("_")[0]
                                label = label_map.get(label, -1)
                                if label != -1:
                                    self.images.append(img)
                                    self.labels.append(label)
                                # self.labels.append(lable_map[name])
                            elif file_path.endswith(".tif"):
                                # here are the reading of tif files                        
                                try:
                                    image_array = tifffile.imread(file_path)
                                except TypeError:
                                    pass
                                    # print(e)
                                img_rescaled = 255 * (image_array - image_array.min()) / (image_array.max() - image_array.min())
                                # dont think this is necessary, can be commented out
                            
                                #My Comments, June 25th: the above comment refers to the line img_col = cv2.applyColorMap...
                                #this was originally commented out, but when I ran the code I got an error saying
                                #img_col is not defined, so it seems that I at least need this commented back in
                                img_col = cv2.applyColorMap(img_rescaled.astype(np.uint8), cv2.COLORMAP_DEEPGREEN)
                                img = Image.fromarray(img_col)
                                img = img.convert("RGB")
                                name = file.split("_")[0]
                                label = label_map.get(label, -1)
                                if label != -1:
                                    self.images.append(img)
                                    self.labels.append(label)
                                break
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img = self.images[idx]
        label = self.labels[idx]
        # apply transformation         
        if self.transform is not None:
            img = self.transform(img)
        return img, label

#My Comments, June 25th:
#trains the model based on loss values; interestingly, it seems as though instead
#of returning the model after the last epoch, it chooses the model from the epoch in 
#which it performed best
#-model is the neural network model we are using
#-criterion is I believe how we measure loss, e.g. 'cross-entropy'
#optimizer is I guess something like in TensorFlow, where the choices were e.g. 'adam'
#-dataloaders is I believe the data separated into training and evaluation
def train_model(model, criterion, optimizer, dataloaders, num_epochs=50, debug=False):
    since = time.time()

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    loss_values = []
    for epoch in range(num_epochs+1):
        if epoch % 5 == 0 and debug:
            print(f'Epoch {epoch}/{num_epochs}')
            
    
        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0
            total = 0
            # Iterate over data.
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # zero the parameter gradients
                optimizer.zero_grad()

                # forward
                # track history if only in train
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs.data, 1)
                    loss = criterion(outputs, labels)

                    # backward + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # statistics
                running_loss += loss.item()
                running_corrects += (preds == labels).sum().item()
                total += labels.size(0)

            epoch_loss = running_loss / len(dataloaders[phase])
            loss_values.append(epoch_loss)
            epoch_acc =  running_corrects / total
            if epoch % 5 == 0 and debug:
                print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')   
                

            # deep copy the model
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
        if epoch % 5 == 0 and debug:  
            print('-' * 10)

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:4f}')

    # load best model weights
    model.load_state_dict(best_model_wts)
    return model, loss_values

def get_metrics(model, test_dataloader):
    true_labels = []
    predicted_labels = []
    accuracy_values = []
    correct = 0
    total = 0
    model.eval()

    with torch.no_grad():
        for images, labels in test_dataloader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)

            # true_labels.extend(labels.numpy())
            # predicted_labels.extend(predicted.numpy())
            true_labels.extend(labels.cpu().numpy())
            predicted_labels.extend(predicted.cpu().numpy())

    precision = precision_score(true_labels, predicted_labels, average='macro')
    recall = recall_score(true_labels, predicted_labels, average='macro')
    f1 = f1_score(true_labels, predicted_labels, average='macro')
    accuracy = accuracy_score(true_labels, predicted_labels)
    # print(f"accuracy: {accuracy}")
    # print(f"Precision: {precision}")
    # print(f"Recall: {recall}")
    # print(f"F1 Score: {f1}")
    return accuracy, precision, recall, f1

label_map = {"BMP4" :0, "CHIR": 1, "DS": 2, "DS+CHIR": 3,  "WT": 4}
# Define the input shape of the images
input_shape = (3, 224, 224)
# Define the number of classes
num_classes = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#model = models.googlenet(pretrained=True)

# Define the data transformations
transform = transforms.Compose([
    transforms.Resize(input_shape[1:]),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])
# Load the dataset

#June 25th, My Comments: Changed the file path for my computer
#August 16, My Comments: Include option for dataloader to be
#completely test data
def get_dataloaders(time, randCat, train = True):
    dataset = MyDataset('C:/Users/aruys/Dropbox (GaTech)/CZIConverted', randCat, transform=transform, timestamp=time)
    
    if train:
        train_dataset, test_dataset = train_test_split(dataset, test_size=0.2, random_state=42)
        # Create data loaders for train and test sets
    
        train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True)
        test_dataloader = DataLoader(test_dataset, batch_size=8, shuffle=False)
        dataloaders = {'train': train_dataloader, 'val': test_dataloader}
        return dataloaders
    else:
        test_dataloader = DataLoader(dataset, batch_size=8,shuffle = False)
        return test_dataloader
        
#June 26th, My Comments: since we are switching to ResNet we now have 512 features instead
#of 1024
#model.fc = nn.Linear(512, num_classes)
# Define the loss function and optimizer
#criterion = nn.CrossEntropyLoss()
#optimizer = optim.SGD(model.fc.parameters(), lr=0.001, momentum=0.9)
#model = model.to(device)

def plot_metric(lst, title, line=''):
    # x_values = [i + 1 for i in range(len(lst))]
    # x_values = [i * 50 for i in range(1, len(accuracy_list)//50 + 1)]
    # Plot the accuracy values
    
    #June 25th, My Comments: Curly trained the model at every 
    #timepoint, so here she subsampled so that only one out of
    #every ten timepoints were graphed. Since I only trained the model
    #on one out of every ten timepoints, I need ot get rid of this
    #subsampling
    #x_values = [i + 1 for i in range(len(lst)) if i % 10 == 0]
    #lst = [lst[i] for i in range(len(lst)) if i % 10 == 0]
    
    x_values = [10*i for i in range(len(lst))]
    lst = [lst[i] for i in range(len(lst))]
    
    figl = plt.figure()
    axl = figl.add_axes([0,0,1,1])
    
    axl.plot(x_values, lst, marker='o', linestyle=line)
    axl.set_ylim(0, 1)
    # Set the x-axis label and title
    axl.set_xlabel('Timestamp')
    axl.set_ylabel(title)
    axl.set_title('ResNet')
    # Define the desired tick locations and labels
    xtick_locations = np.arange(0, 300, 50)
    xtick_labels = [str(x) for x in xtick_locations]

    # Set the x-axis tick locations and labels
    axl.set_xticks(xtick_locations, xtick_labels)
    # plt.xticks(range(1, len(accuracy_list) + 1))
    # Display the plot
    #plt.show()
    figl.savefig(title,bbox_inches='tight')

In [10]:
###August 23, 2023: This trains the model for the ``fixed testing'', don't need this now


###      import sys
###      criterion = nn.CrossEntropyLoss()
###      optimizer = optim.SGD(model.fc.parameters(), lr=0.001, momentum=0.9)
###      original_stderr = sys.stderr
###      accuracy_dict = {}
###      precision_dict = {}
###      recall_dict = {}
###      f1_dict = {}
###      sys.stderr = open(os.devnull, 'w')
###      #for timestamp in range(1, 290):
###      #myRange = [1]+list(range(10, 280, 10))+[288]


###      dataloaders = get_dataloaders(tmptOfInterest)
###      model_trained, loss_values = train_model(model, criterion, optimizer, dataloaders, num_epochs=10)
###      #torch.save(model_trained.state_dict(), 'ResNet_'+tOI+'_'+date+'.pt')
###      accuracy, precision, recall, fOne = get_metrics(model_trained, dataloaders['val'])
###      accuracy_dict[tOI]=accuracy
###      precision_dict[tOI]=precision
###      recall_dict[tOI]=recall
###      f1_dict[tOI]=fOne
###      sys.stderr = original_stderr
###      timestamp:  1

###      ##plot_metric(accuracy_list, 'ResNet_Accuracy_'+date)
###      ##plot_metric(precision_list, 'ResNet_Precision_'+date)
###      ##plot_metric(recall_list, 'ResNet_Recall_'+date)
###      ##plot_metric(f1_list, 'ResNet_F1 Score_'+date)


In [4]:
def evalFixedTrain(t,randCat):
    accuracy_dict = {'x':[], 'fix_y':[], 'flo_y':[], 'diff_y':[]}
    precision_dict = {'x':[], 'fix_y':[], 'flo_y':[], 'diff_y':[]}
    recall_dict = {'x':[], 'fix_y':[], 'flo_y':[], 'diff_y':[]}
    f1_dict = {'x':[], 'fix_y':[], 'flo_y':[], 'diff_y':[]}

    tmpts = [1]+list(range(10,290,10))+[288]
    
    #model_fixed = models.resnet18(pretrained=True)
    #model_fixed.fc = nn.Identity()
    #model_fixed.fc = nn.Linear(512, num_classes)
    #model_fixed.load_state_dict(torch.load('ResNets_Trained/'+findNet(t,randCat)))
    #model_fixed = model_fixed.to(device)
    model_fixed = torch.load('TRAIN_ResNets_Trained/'+randCat+'/'+findNet(t,randCat))
    model_fixed.eval()
    tmpts = [1]+list(range(10,290,10))+[288]
    for i in tmpts:
        #run through all the timepoints, including the fixed one now
        fix_acc, fix_pre, fix_rec, fix_f1 = get_metrics(model_fixed, get_dataloaders(i,randCat,train=False))
        recall_dict['x'].append(i)
        precision_dict['x'].append(i)
        accuracy_dict['x'].append(i)
        f1_dict['x'].append(i)
        recall_dict['fix_y'].append(fix_acc)
        precision_dict['fix_y'].append(fix_pre)
        accuracy_dict['fix_y'].append(fix_rec)
        f1_dict['fix_y'].append(fix_f1)
        #now compare with a model trained on that timepoint
        #offset by one (e.g. train on timepoint t020, test on timepoint 010) so we have testing on full dataset for
        #our metrics
        if i==1:
            j=10
        elif i <280:
            j = i+10
        elif i == 280:
            j = 288
        else:
            j = 280
        #model_float = models.resnet18(pretrained=True)
        #model_float.fc = nn.Identity()
        #model_float.fc = nn.Linear(512, num_classes)
        #model_float.load_state_dict(torch.load('ResNets_Trained/'+findNet(j,randCat)))
        #model_float = model_float.to(device)
        model_float = torch.load('TRAIN_ResNets_Trained/'+randCat+'/'+findNet(j,randCat))
        model_float.eval()
        flo_acc, flo_pre, flo_rec, flo_f1 = get_metrics(model_float, get_dataloaders(i,randCat,train=False))
        recall_dict['flo_y'].append(flo_acc)
        precision_dict['flo_y'].append(flo_pre)
        accuracy_dict['flo_y'].append(flo_rec)
        f1_dict['flo_y'].append(flo_f1)
        recall_dict['diff_y'].append(flo_acc-fix_acc)
        precision_dict['diff_y'].append(flo_pre-fix_pre)
        accuracy_dict['diff_y'].append(flo_rec-fix_rec)
        f1_dict['diff_y'].append(flo_f1-fix_f1)
    return accuracy_dict, precision_dict, recall_dict, f1_dict



In [3]:
def plot_tmpt_diffs(metric_dict, metric_name,randCat):
    
    figl = plt.figure()
    axl = figl.add_axes([0,0,1,1])
    
    #axl.plot(accuracy_dict['x'], accuracy_dict['y'], marker='o', linestyle=line)
    axl.scatter(metric_dict['x'], metric_dict['fix_y'])
    #axl.scatter([tmptOfInterest],[metric_dict[tOI]],marker = 'x', color = 'r')
    axl.set_ylim(0, 1)
    # Set the x-axis label and title
    axl.set_xlabel('Timepoint')
    axl.set_ylabel(metric_name +'for Fixed Training')
    axl.set_title(metric_name +' - '+netType+' Trained on '+tOI)
    # Define the desired tick locations and labels
    #xtick_locations = np.arange(0, 300, 50)
    #xtick_labels = [str(x) for x in xtick_locations]

    # Set the x-axis tick locations and labels
    #axl.set_xticks(xtick_locations, xtick_labels)
    #plt.xticks(range(1, len(accuracy_list) + 1))
    #Display the plot
    #plt.show()
    #figl.savefig('DIFF_TmptDiffs/'+randCat+'/Fix_'+metric_name+'_'+netType+'_Trained_on_'+tOI+'.jpg',bbox_inches='tight')
    #figl.close()

    fign = plt.figure()
    axn = fign.add_axes([0,0,1,1])
    
    #axl.plot(accuracy_dict['x'], accuracy_dict['y'], marker='o', linestyle=line)
    axn.scatter(metric_dict['x'], metric_dict['flo_y'])
    #axn.scatter([tmptOfInterest],[metric_dict[tOI]],marker = 'x', color = 'r')
    axn.set_ylim(0, 1)
    # Set the x-axis label and title
    axn.set_xlabel('Timepoint')
    axn.set_ylabel(metric_name +'for Floating Training')
    axn.set_title(metric_name +' - '+netType +' Trained on Timepoint Offset by One')
    # Define the desired tick locations and labels
    #xtick_locations = np.arange(0, 300, 50)
    #xtick_labels = [str(x) for x in xtick_locations]

    # Set the x-axis tick locations and labels
    #axl.set_xticks(xtick_locations, xtick_labels)
    #plt.xticks(range(1, len(accuracy_list) + 1))
    #Display the plot
    #plt.show()
    #fign.savefig('DIFF_TmptDiffs/'+randCat+'/Flo_'+metric_name+'_'+netType+'_Trained_on_'+tOI+'.jpg',bbox_inches='tight')
    #figl.close()
               
                   
                   
    figm = plt.figure()
    axm = figm.add_axes([0,0,1,1])
    
    #axl.plot(accuracy_dict['x'], accuracy_dict['y'], marker='o', linestyle=line)
    axm.scatter(metric_dict['x'], metric_dict['diff_y'])
    #axm.scatter([tmptOfInterest],[metric_dict[tOI]],marker = 'x', color = 'r')
    axm.set_ylim(-0.5, 0.5)
    # Set the x-axis label and title
    axm.set_xlabel('Timepoint')
    axm.set_ylabel(metric_name+' Differential')
    axm.set_title('Timepoint Differential '+ metric_name +' - '+netType+' Trained on '+tOI)
    # Define the desired tick locations and labels
    #xtick_locations = np.arange(0, 300, 50)
    #xtick_labels = [str(x) for x in xtick_locations]

    # Set the x-axis tick locations and labels
    #axl.set_xticks(xtick_locations, xtick_labels)
    #plt.xticks(range(1, len(accuracy_list) + 1))
    #Display the plot
    #plt.show()
    #figm.savefig('DIFF_TmptDiffs/'+randCat+'/Diff_'+metric_name+'_'+netType+'_Trained_on_'+tOI+'.jpg',bbox_inches='tight')
    #figm.close()

In [5]:
#timepoints = [1]+[288]+list(range(10,150,10))+list(range(170, 290, 10))
timepoints = [1]+[288]+list(range(10,290,10))
#timepoints = list(range(10,20,10))
randCat = 'RandCat1'
netType = 'ResNet'
for tmptOfInterest in timepoints:
    tOI = strTmpt(tmptOfInterest)
    acc, pre, rec, f1 = evalFixedTrain(tmptOfInterest,randCat)

    #plot_tmpt_diffs(acc, 'Accuracy',randCat)
    #plot_tmpt_diffs(f1,'F1')
    #plot_tmpt_diffs(pre, 'Precision')
    #plot_tmpt_diffs(rec, 'Recall')

    gud = {'Timepoint': acc['x'], 'Fixed Accuracy':acc['fix_y'], 'Floating Accuracy':acc['flo_y'], 'Diff. Accuracy':acc['diff_y'],\
       'Fixed Precision':pre['fix_y'], 'Floating Precision':pre['flo_y'], 'Diff. Precision':pre['diff_y'],\
       'Fixed Recall':rec['fix_y'], 'Floating Recall':rec['flo_y'], 'Diff. Recall':rec['diff_y'],\
       'Fixed F1':f1['fix_y'], 'Floating F1':f1['flo_y'], 'Diff. F1':f1['diff_y']}
    gudf = pd.DataFrame(data = gud)
    gudf.to_csv('DIFF_TmptDiffs/'+randCat+'/'+netType+'_Trained_on_'+tOI+'.csv', index = False)

#hud = {'Val. Accuracy':[accuracy], 'Val. Precision': [precision], 'Val Recall':[recall],\
#      'Val F1':[fOne]}
#hudf = pd.DataFrame(data = hud)
#hudf.to_csv('Timepoint_Differentials/Metrics/'+netType+'_Trained_and_Tested_on_'+tOI+'_'+today+'.csv', index = False)

C:\Users\aruys\anaconda3\envs\myEnv\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\aruys\anaconda3\envs\myEnv\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\aruys\anaconda3\envs\myEnv\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\aruys\anaconda3\envs\myEnv\Lib\site-packages\sklearn